In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import random

In [2]:
import numpy as np

def conv2d(image, kernel, stride, padding):
    # Get dimensions
    image_height, image_width = image.shape
    kernel_height, kernel_width = kernel.shape

    # Apply padding
    if padding > 0:
        padded_image = np.pad(image, ((padding, padding), (padding, padding)), mode='constant')
    else:
        padded_image = image

    # Get padded image dimensions
    padded_image_height, padded_image_width = padded_image.shape

    # Calculate output dimensions
    # H_out = floor((H_in - H_k + 2*P) / S) + 1  --> Here H_in already includes padding if any
    output_height = (padded_image_height - kernel_height) // stride + 1
    output_width = (padded_image_width - kernel_width) // stride + 1

    # Initialize output feature map
    feature_map = np.zeros((output_height, output_width))

    # Perform convolution
    for i in range(output_height):
        for j in range(output_width):
            # Extract the image patch
            row_start = i * stride
            col_start = j * stride
            image_patch = padded_image[row_start : row_start + kernel_height,
                                         col_start : col_start + kernel_width]

            # Element-wise multiplication and sum
            feature_map[i, j] = np.sum(image_patch * kernel)

    return feature_map

# Test Image
image = np.array([
    [3, 1, 0, 2, 4],
    [1, 5, 3, 2, 1],
    [0, 2, 6, 4, 3],
    [2, 3, 1, 5, 2],
    [1, 0, 2, 3, 4]
])

# Sobel-X Kernel
sobel_x_kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

# Parameters
stride = 1
padding = 0

# Apply convolution
output_feature_map = conv2d(image, sobel_x_kernel, stride, padding)

print("Input Image Shape:", image.shape)
print("Kernel Shape:", sobel_x_kernel.shape)
print("\nOutput Feature Map (Sobel-X):")
print(output_feature_map)

# Verify output shape manually
input_height, input_width = image.shape
kernel_height, kernel_width = sobel_x_kernel.shape

expected_output_height = (input_height - kernel_height) // stride + 1
expected_output_width = (input_width - kernel_width) // stride + 1

print(f"\nExpected Output Shape: ({expected_output_height}, {expected_output_width})")
print("Actual Output Shape:", output_feature_map.shape)

assert output_feature_map.shape == (expected_output_height, expected_output_width)
print("Output shape verification successful!")

Input Image Shape: (5, 5)
Kernel Shape: (3, 3)

Output Feature Map (Sobel-X):
[[ 7. -3. -3.]
 [13.  3. -7.]
 [ 5.  9.  1.]]

Expected Output Shape: (3, 3)
Actual Output Shape: (3, 3)
Output shape verification successful!


### Output Size Derivation

The formula for calculating the output spatial dimension (height or width) is:

`Output = floor(((Input - Kernel + 2 * Padding) / Stride)) + 1`

Let's apply this formula to each scenario:

---

**(a) Input: 28×28, Kernel: 5×5, Padding: 0 (valid), Stride: 1**

*   Input Size (H_in, W_in) = 28
*   Kernel Size (H_k, W_k) = 5
*   Padding (P) = 0
*   Stride (S) = 1

**Height Calculation:**
H_out = floor(((28 - 5 + 2 * 0) / 1)) + 1
H_out = floor((23 / 1)) + 1
H_out = 23 + 1
H_out = 24

**Width Calculation:**
W_out = floor(((28 - 5 + 2 * 0) / 1)) + 1
W_out = floor((23 / 1)) + 1
W_out = 23 + 1
W_out = 24

**Output Size: 24×24**

---

**(b) Input: 28×28, Kernel: 3×3, Padding: 1 (same), Stride: 1**

*   Input Size (H_in, W_in) = 28
*   Kernel Size (H_k, W_k) = 3
*   Padding (P) = 1 (for 'same' padding with stride 1, P is typically chosen to make output size equal to input size)
*   Stride (S) = 1

**Height Calculation:**
H_out = floor(((28 - 3 + 2 * 1) / 1)) + 1
H_out = floor(((25 + 2) / 1)) + 1
H_out = floor((27 / 1)) + 1
H_out = 27 + 1
H_out = 28

**Width Calculation:**
W_out = floor(((28 - 3 + 2 * 1) / 1)) + 1
W_out = floor(((25 + 2) / 1)) + 1
W_out = floor((27 / 1)) + 1
W_out = 27 + 1
W_out = 28

**Output Size: 28×28**

---

**(c) Input: 32×32, Kernel: 3×3, Padding: 0 (valid), Stride: 2**

*   Input Size (H_in, W_in) = 32
*   Kernel Size (H_k, W_k) = 3
*   Padding (P) = 0
*   Stride (S) = 2

**Height Calculation:**
H_out = floor(((32 - 3 + 2 * 0) / 2)) + 1
H_out = floor((29 / 2)) + 1
H_out = floor(14.5) + 1
H_out = 14 + 1
H_out = 15

**Width Calculation:**
W_out = floor(((32 - 3 + 2 * 0) / 2)) + 1
W_out = floor((29 / 2)) + 1
W_out = floor(14.5) + 1
W_out = 14 + 1
W_out = 15

**Output Size: 15×15**

---

**(d) Two consecutive Conv2D layers:**

*   **Layer 1:** Input: 32×32, K=3, P=1, S=1

    *   Input Size (H_in, W_in) = 32
    *   Kernel Size (H_k, W_k) = 3
    *   Padding (P) = 1
    *   Stride (S) = 1

    **Height Calculation (Layer 1):**
    H_out1 = floor(((32 - 3 + 2 * 1) / 1)) + 1
    H_out1 = floor(((29 + 2) / 1)) + 1
    H_out1 = floor((31 / 1)) + 1
    H_out1 = 31 + 1
    H_out1 = 32

    **Width Calculation (Layer 1):**
    W_out1 = floor(((32 - 3 + 2 * 1) / 1)) + 1
    W_out1 = floor(((29 + 2) / 1)) + 1
    W_out1 = floor((31 / 1)) + 1
    W_out1 = 31 + 1
    W_out1 = 32

    **Output of Layer 1: 32×32**

*   **Layer 2:** Input: (Output of Layer 1) 32×32, K=3, P=0, S=1

    *   Input Size (H_in, W_in) = 32
    *   Kernel Size (H_k, W_k) = 3
    *   Padding (P) = 0
    *   Stride (S) = 1

    **Height Calculation (Layer 2):**
    H_out2 = floor(((32 - 3 + 2 * 0) / 1)) + 1
    H_out2 = floor((29 / 1)) + 1
    H_out2 = 29 + 1
    H_out2 = 30

    **Width Calculation (Layer 2):**
    W_out2 = floor(((32 - 3 + 2 * 0) / 1)) + 1
    W_out2 = floor((29 / 1)) + 1
    W_out2 = 29 + 1
    W_out2 = 30

**Final Output Size: 30×30**

In [3]:
import tensorflow as tf

# LeNet-5 Architecture
lenet5_model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(6, (5, 5), activation='tanh', input_shape=(28, 28, 1), padding='valid'),
    tf.keras.layers.AveragePooling2D((2, 2), strides=(2, 2)),
    tf.keras.layers.Conv2D(16, (5, 5), activation='tanh', padding='valid'),
    tf.keras.layers.AveragePooling2D((2, 2), strides=(2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(120, activation='tanh'),
    tf.keras.layers.Dense(84, activation='tanh'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Print model summary
lenet5_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 12, 12, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 16)       │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 4, 4, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,426 (173.54 KB)

 Trainable params: 44,426 (173.54 KB)

 Non-trainable params: 0 (0.00 B)

### (b) Manual Parameter Calculation for the First Conv2D layer

The formula for calculating parameters in a convolutional layer is: `(K × K × C_in + 1) × C_out`

Where:
*   `K` = Kernel Size (height or width)
*   `C_in` = Number of input channels
*   `1` = Bias term for each filter
*   `C_out` = Number of output channels (or filters)

For the first `Conv2D` layer in LeNet-5:
*   `K` = 5 (kernel size is 5x5)
*   `C_in` = 1 (input image has 1 channel for grayscale)
*   `C_out` = 6 (number of filters)

Substituting these values into the formula:
Parameters = `(5 × 5 × 1 + 1) × 6`
Parameters = `(25 + 1) × 6`
Parameters = `26 × 6`
Parameters = `156`

This matches the parameter count for the first `conv2d` layer shown in the model summary.

### (c) Explanation of AvgPooling vs. MaxPooling

AvgPooling in LeNet-5:
Used to downsample by taking the average of values, producing smoother feature maps and reducing dimensions while preserving general information.

MaxPooling Today:
More common because it keeps the strongest features (maximum values), improves performance, adds translation invariance, and reduces computation more effectively.

In [4]:
import tensorflow as tf

def create_custom_cifar10_cnn():
    model = tf.keras.Sequential([
        # Convolutional Block 1
        tf.keras.layers.Conv2D(64, (3, 3), padding='same', input_shape=(32, 32, 3)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Convolutional Block 2
        tf.keras.layers.Conv2D(128, (3, 3), padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Convolutional Block 3
        tf.keras.layers.Conv2D(256, (3, 3), padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Classification Head
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    return model

# Create and print the model summary
custom_cifar10_model = create_custom_cifar10_cnn()
custom_cifar10_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 440,970 (1.68 MB)

 Trainable params: 440,074 (1.68 MB)

 Non-trainable params: 896 (3.50 KB)

### Custom CNN Architecture for CIFAR-10

**ASCII Diagram:**

```
Input (32x32x3)
    |
    V
Conv2D (64 filters, 3x3, padding='same')
    |
    V
BatchNormalization
    |
    V
ReLU
    |
    V
MaxPooling2D (2x2)
    |
    V
Conv2D (128 filters, 3x3, padding='same')
    |
    V
BatchNormalization
    |
    V
ReLU
    |
    V
MaxPooling2D (2x2)
    |
    V
Conv2D (256 filters, 3x3, padding='same')
    |
    V
BatchNormalization
    |
    V
ReLU
    |
    V
GlobalAveragePooling2D
    |
    V
Dense (256 units, activation='relu')
    |
    V
Dropout (0.5)
    |
    V
Dense (10 units, activation='softmax')
```

**Design Rationale:**
This architecture leverages three convolutional blocks to progressively extract hierarchical features from the CIFAR-10 images. Each block incorporates `BatchNormalization` for faster and more stable training, followed by `ReLU` for non-linearity. `MaxPooling` layers are used to reduce spatial dimensions, increasing the receptive field. A `GlobalAveragePooling2D` layer replaces `Flatten` to reduce the number of parameters and combat overfitting by averaging feature maps into a single vector. Finally, a `Dropout` layer is included in the dense classification head to further enhance regularization and prevent overfitting, leading to a robust model suitable for CIFAR-10 classification within the target parameter range.

Questions:

Q1.Compare the parameter efficiency of two stacked 3×3 Conv layers versus one 5×5 Conv layer on the same input with the same number of filters. Which uses fewer parameters? Show numerical proof and explain any other advantages of the smaller kernel approach.

ANS. Assume:

Input channels = C           
Output filters = F

Single 5×5 Conv layer:         
Parameters = (5 × 5 × C + 1) × F = (25C + 1)F           
Two stacked 3×3 Conv layers:           
First layer: (3 × 3 × C + 1) × F = (9C + 1)F      
Second layer: (3 × 3 × F + 1) × F = (9F + 1)F        
Total ≈ (9C + 9F + 2)F

Additional Advantages of 3×3 Stacking   
Same receptive field   
Two 3×3 ≈ one 5×5   
More non-linearity   
Two activation layers → better feature learning  
Better performance   
Deeper networks capture more complex patterns    
Reduced overfitting   
Fewer parameters → less risk    

Q2. What is the role of Batch Normalisation in a CNN? Where in the layer stack should it be placed (before or after activation), and why? Mention at least two empirical benefits it provides during training.

ANS. Role of Batch Normalisation (BN):

Normalises layer inputs to have mean ≈ 0 and variance ≈ 1
Stabilises training and reduces internal covariate shift

Placement:

Applied after Conv layer and before activation     
Order: Conv → BatchNorm → ReLU   
Reason:
BN normalises raw outputs before non-linearity and
leads to more stable and effective activation

Empirical Benefits:

Faster convergence-
Allows higher learning rates   
Improved stability-
Reduces exploding/vanishing gradients   
Regularisation effect-
Slightly reduces overfitting   
Better performance-
Often improves validation accuracy

Q3. Your custom CNN has a GlobalAveragePooling layer before the Dense head. What does this layer do geometrically? What would happen to the parameter count and spatial information if you replaced it with Flatten?

ANS. Function of GlobalAveragePooling layer:

Converts each feature map into a single value by averaging over H×W      
Geometrically: collapses spatial dimensions (H, W) → keeps only channel-wise information     
Output shape becomes (C) instead of (H × W × C)

If replaced with Flatten:
Parameter count:   
Flatten outputs H × W × C values  
Leads to very large Dense layers → huge increase in parameters

Spatial information:    
Flatten preserves spatial details explicitly     
But treats all pixels independently (no spatial structure awareness)